# Can we predict `claim_within_year`?
Plan (per checklist.md): hold out a test set first; cross-validate inside train only; preprocessing inside folds; candidates named before scoring; choice written down before the single test score.

Data notes: no date column, so row order is not treated as time. `device_serial` is unique per row (no repeated entities), so a stratified random split is valid. `device_serial` is an identifier and is dropped. All other columns are recorded at sale/registration, i.e. known at prediction time.

In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score
df = pd.read_csv('data.csv')
assert df.device_serial.is_unique
y = df['claim_within_year']; X = df.drop(columns=['claim_within_year','device_serial'])
print(df.shape, y.mean())

## 1. Test set (set aside before any modelling)

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=0)
print(len(X_tr), len(X_te), y_tr.mean(), y_te.mean())

## 2. Candidate models (named before scoring)
1. `baseline`: predicts the training prevalence.
2. `logreg`: one-hot channel + scaled numerics, logistic regression (C in {0.1, 1}).
3. `hgb`: HistGradientBoosting with one-hot channel (max_depth in {2, 3}, learning_rate 0.05).

Selection metric: CV ROC-AUC (threshold-free). Decision threshold: chosen on out-of-fold predictions of the winner to maximise F1. Preprocessing lives in the pipelines, so it is refit in every fold.

In [ ]:
num = ['device_age_years','price_eur','daily_hours']; cat = ['channel']
def prep(scale):
    return ColumnTransformer([('n', StandardScaler() if scale else 'passthrough', num),
                              ('c', OneHotEncoder(handle_unknown='ignore'), cat)])
cands = {'baseline': Pipeline([('p',prep(False)),('m',DummyClassifier(strategy='prior'))])}
for C_ in (0.1,1): cands[f'logreg_C{C_}'] = Pipeline([('p',prep(True)),('m',LogisticRegression(C=C_,max_iter=1000))])
for d in (2,3): cands[f'hgb_d{d}'] = Pipeline([('p',prep(False)),('m',HistGradientBoostingClassifier(max_depth=d,learning_rate=0.05,max_iter=200,random_state=0))])
cv = StratifiedKFold(5, shuffle=True, random_state=0)   # entities are unique rows
oof = {}; rows=[]
for k,m in cands.items():
    oof[k] = cross_val_predict(m, X_tr, y_tr, cv=cv, method='predict_proba')[:,1]
    rows.append((k, roc_auc_score(y_tr,oof[k]), average_precision_score(y_tr,oof[k])))
cvres = pd.DataFrame(rows, columns=['model','cv_auc','cv_ap']).sort_values('cv_auc',ascending=False); cvres

## 3. Choice on CV only (written before test scoring)

In [ ]:
best = cvres.iloc[0].model
ths = np.linspace(0.02,0.6,59)
f1s = [f1_score(y_tr, oof[best]>=t) for t in ths]
thr = float(ths[int(np.argmax(f1s))])
print('chosen model:', best, '| threshold:', round(thr,3), '| CV F1:', round(max(f1s),3))

**Decision recorded:** the model and threshold printed above are final; the test set is now scored exactly once, with no further changes.

In [ ]:
final = cands[best].fit(X_tr, y_tr)
p = final.predict_proba(X_te)[:,1]
base = cands['baseline'].fit(X_tr,y_tr).predict_proba(X_te)[:,1]
print(f'TEST ROC-AUC {roc_auc_score(y_te,p):.3f}  AP {average_precision_score(y_te,p):.3f}  F1@thr {f1_score(y_te,p>=thr):.3f}')
print(f'(baseline AP = prevalence {average_precision_score(y_te,base):.3f})')
rng=np.random.default_rng(0); idx=np.arange(len(y_te)); bs=[]
for _ in range(1000):
    i=rng.choice(idx,len(idx));
    if y_te.iloc[i].nunique()>1: bs.append(roc_auc_score(y_te.iloc[i],p[i]))
print('AUC 95% bootstrap CI', np.percentile(bs,[2.5,97.5]).round(3))

## Conclusion
See the printed test metrics above: the headline is the single test-set score of the CV-chosen model, compared with the prevalence baseline (AUC 0.5).